In [ ]:
%%capture
import os, re
# تثبيت المكتبات الأساسية (Libraries) عشان يشتغل الموديل
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # التثبيت لو كنت شغال على جهازك الخاص (Local setup)
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

# ثبتنا نسخ محددة (Specific versions) من الـ (Transformers) والـ (TRL)
# السبب: عشان نضمن استقرار الموديل مع اللغة العربية وما تطلع لك كلمات مكسرة (Tokens error)
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [1]:
from unsloth import FastLanguageModel
import torch
# تحديد أقصى طول للنص (Sequence length) ونوع البيانات (Data type)
max_seq_length = 2048
dtype = None
# تفعيل تقنية الـ (4-bit quantization) لتقليل استهلاك الذاكرة (VRAM) بنسبة كبيرة
load_in_4bit = True

# تحميل الموديل المختار (Llama-3.1-8B) مع المحول النصي (Tokenizer)
# ملاحظة: غيرنا الموديل لـ (Llama-3.1) لأنه أقوى بكثير في اللغة العربية (Arabic language)
# وما يغلط في المصطلحات اللغوية (Vocabulary) مثل الموديلات الأصغر.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.11: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    # تحديد قيمة الـ (Rank - r) وقوة التأثير (lora_alpha) بـ 16
    # ملاحظة: خليناها 16 لأن موديل (Llama-3.1) ضخم وذكي (Robust) وما يحتاج قيم عالية عشان يفهم
    r = 16,
    # اختيار الطبقات (Target modules) اللي بنسوي لها تدريب (Fine-tuning)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # الـ (Dropout) صفر يخلي التدريب أسرع (Optimized)
    bias = "none",    # عدم استخدام (Bias) يقلل استهلاك الذاكرة (VRAM)
    # تقنية (Gradient checkpointing) من (Unsloth) توفر مساحة كبيرة في الذاكرة
    use_gradient_checkpointing = "unsloth",
    random_state = 3407, # تثبيت الرقم العشوائي لضمان نفس النتائج (Reproducibility)
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.3.11 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [3]:
# تحديد القالب (Prompt Template) اللي بيستخدمه الموديل في الحوار
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}""" # ملاحظة: استخدمنا كلمة (Response) بدل (output) لأن الموديلات تفهمها كإشارة (Trigger) للرد

EOS_TOKEN = tokenizer.eos_token # استدعاء رمز النهاية (End Of String) لإيقاف الموديل عن الكلام

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        # دمج السؤال والجواب في القالب (Format) وإضافة رمز النهاية (EOS)
        # السبب: بدون رمز النهاية الموديل بيكرر الأسئلة (Looping) وما يعرف متى يسكت
        text = alpaca_prompt.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset
# تحميل ملف البيانات (Dataset) بصيغة (JSON)
dataset = load_dataset("json", data_files="tuwaiq_dataset_100.json", split = "train")
# تطبيق دالة التنسيق (Map) على كامل البيانات لتجهيزها للتدريب (Fine-tuning)
dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/249 [00:00<?, ? examples/s]

In [4]:
from trl import SFTConfig, SFTTrainer

# إعدادات التدريب (Training Arguments) وتحديد كيف بيذاكر الموديل
training_args = SFTConfig(
    per_device_train_batch_size = 2,      # حجم البيانات في كل خطوة لتقليل الضغط على الذاكرة (VRAM)
    gradient_accumulation_steps = 4,      # تجميع الخطوات لتحديث الأوزان (Batch size) بشكل فعّال ومستقر
    warmup_steps = 10,                    # زيادة سرعة التعلم تدريجياً (Warmup) عشان ما ينهز الموديل في البداية

    # تحديد عدد الدورات (Epochs) بدل الخطوات الجامدة (Steps)
    # ملاحظة: اخترنا 4 دورات عشان نضمن إنه يمر على الـ 200 سؤال كاملة ويتشربها صح
    num_train_epochs = 4,
    dataset_num_proc = 2,                 # تقسيم العمل على معالجين (Processors) لتسريع التجهيز

    # سرعة التعلم (Learning rate) ونوع الجدولة (Scheduler)
    # ملاحظة: قللنا السرعة لـ 5e-5 واستخدمنا (cosine) عشان التعلم يكون أدق وما يسبب "هلوسة" لغوية
    learning_rate = 5e-5,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,                    # مراقبة نسبة الخطأ (Loss) خطوة بخطوة
    optim = "adamw_8bit",                 # محسن (Optimizer) موفر للذاكرة يسرع العملية
    weight_decay = 0.01,                  # تقنية لمنع الحفظ الصم (Overfitting) وزيادة المرونة
    lr_scheduler_type = "cosine",
    seed = 3407,                          # تثبيت العشوائية لضمان تكرار النتائج
    output_dir = "outputs",               # مكان حفظ ملفات التدريب (Checkpoints)
    report_to = "none",
)

# إنشاء المدرب (Trainer) وربط الموديل بالبيانات (Dataset)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,                      # عدم دمج النصوص لضمان جودة فصل الأسئلة عن الأجوبة
    args = training_args,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/249 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [5]:
# كود فحص استهلاك الذاكرة (Memory Monitoring) عشان نراقب سعة كرت الشاشة (GPU)
gpu_stats = torch.cuda.get_device_properties(0) # جلب مواصفات كرت الشاشة المستخدم (Name & Specs)
# حساب الذاكرة المحجوزة حالياً (Reserved memory) وتحويلها لجيجابايت (GB)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
# معرفة إجمالي سعة الذاكرة المتوفرة (Total VRAM) في الجهاز
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

# طباعة النتائج عشان نتأكد من المساحة قبل الزحمة
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

# فائدة هذا الجزء: نضمن وجود مساحة (Buffer) كافية قبل ما يبدأ التدريب (Training)
# والهدف الأساسي هو تجنب توقف البرنامج بسبب نقص الذاكرة (Out of Memory Error).

GPU = Tesla T4. Max memory = 14.563 GB.
6.707 GB of memory reserved.


In [6]:
# لحظة تشغيل "المحرك": هنا يبدأ الموديل عملية التعلم (Fine-tuning) الفعلية
# الموديل بيمر على الـ 200 سؤال ويربط المعلومات ببعضها بناءً على الإعدادات اللي حطيناها
trainer_stats = trainer.train()

# فائدة هذا السطر: تنفيذ عملية التدريب واستخراج الإحصائيات (Training stats)
# مثل نسبة الخطأ (Loss) والوقت المستغرق (Runtime) عشان نعرف جودة التعلم

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 249 | Num Epochs = 4 | Total steps = 128
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,3.057800
2,3.444300
3,3.116000
4,2.987400
5,3.151700
6,3.022600
7,3.032200
8,2.828600
9,2.763400
10,2.710700


In [7]:
# @title عرض الإحصائيات النهائية للذاكرة والوقت (Show final memory and time stats)

# 1. حساب إجمالي الذاكرة المحجوزة (Total Reserved Memory) في كرت الشاشة بعد التدريب
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

# 2. حساب الذاكرة اللي استهلكتها عملية الـ (LoRA) فقط (الفرق بين النهاية والبداية)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)

# 3. حساب النسبة المئوية (Percentage) لإجمالي الذاكرة المستخدمة من سعة الكرت الكلية
used_percentage = round(used_memory / max_memory * 100, 3)

# 4. حساب نسبة استهلاك التدريب (LoRA) فقط من سعة كرت الشاشة
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

# --- طباعة النتائج النهائية للمستخدم (Final results printing) ---

# طباعة الوقت المستغرق للتدريب (Training runtime) بالثواني والدقائق
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)

# طباعة أقصى قدر من الذاكرة (Peak reserved memory) تم حجزه خلال العملية
# ملاحظة: هذي الأرقام تثبت كفاءة (Unsloth) في توفير الموارد (Efficiency)
print(f"Peak reserved memory = {used_memory} GB.")

# طباعة كمية الذاكرة الفعلية اللي احتاجها التدريب (Memory for training)
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")

# طباعة النسب المئوية للاستهلاك (Usage percentage) للتأكد من سلامة كرت الشاشة
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

514.7684 seconds used for training.
8.58 minutes used for training.
Peak reserved memory = 6.707 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 46.055 %.
Peak reserved memory for training % of max memory = 0.0 %.


In [12]:
# تفعيل وضع الاستنتاج (Inference Mode) لزيادة سرعة الرد بنسبة 2x
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# تحديد سؤال المستخدم (User question) وتجهيزه للقالب (Prompt)
user_question = "كيف ممكن اسجل في الاكاديمية؟"

# تحويل النص لرموز رقمية (Tensors) ونقلها لكرت الشاشة (GPU)
inputs = tokenizer([
    alpaca_prompt.format(user_question, "", "")
], return_tensors = "pt").to("cuda")

# كود التوليد (Generation) مع إعدادات تضمن الانضباط اللغوي
outputs = model.generate(
    **inputs,
    max_new_tokens = 64,  # تقليل عدد الكلمات (Tokens) ليركز الموديل على زبدة الإجابة
    use_cache = True,     # استخدام التخزين المؤقت لتسريع التوليد
    temperature = 0.05,   # خفض الحرارة (Temperature) لمنع الهلوسة واختيار الكلمات الأكيدة
    do_sample = False,    # تفعيل البحث المباشر (Greedy Search) لضمان المنطقية (Deterministic)
)

# فك التشفير (Decoding) واستخراج الرد الصافي فقط
decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# طباعة النتيجة النهائية (Final response) بعد تنظيفها من زوائد القالب
if "### Response:" in decoded_output:
    print("الرد النهائي :")
    # فصل النص وأخذ ما بعد كلمة (Response) ليكون الرد احترافي
    print(decoded_output.split("### Response:")[1].strip())

الرد النهائي :
يمكن التسجيل عبر الموقع الالكتروني أو خلال البرامج التدريبية.


In [ ]:
# --- كود حفظ الموديل النهائي (Model Saving) ---

# 1. حفظ الموديل محلياً (Local Save) بصيغة LoRA
# ملاحظة: هذا بيحفظ لك الأوزان اللي تدربت عليها فقط (Adapters)
model.save_pretrained("tuwaiq_model_lora")
tokenizer.save_pretrained("tuwaiq_model_lora")

# 2. تحويل وحفظ الموديل بصيغة GGUF (المثالية للـ Agents)
# ملاحظة: اخترنا (q4_k_m) عشان يكون التوازن ممتاز بين الحجم الصغير والدقة العالية
model.save_pretrained_gguf("tuwaiq_model_gguf", tokenizer, quantization_method = "q4_k_m")

print("تم حفظ الموديل بنجاح! جاهزين للمرحلة الجاية 🚀")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/896 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files:  25%|██▌       | 1/4 [02:15<06:45, 135.32s/it]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]